# LayoutVLM 完整复现 - CVPR 2025

**论文**: [LayoutVLM: Differentiable Optimization of 3D Layout via Vision-Language Models](https://arxiv.org/abs/2412.02193)

**GitHub**: https://github.com/sunfanyunn/LayoutVLM

---

## 🎯 功能说明

此notebook整合了完整的工作流程：
1. ✅ 在Colab中安装Blender 4.2.1
2. ✅ 配置GPU渲染
3. ✅ 运行LayoutVLM生成3D布局
4. ✅ 自动渲染可视化结果

---

## 📋 使用步骤

1. **启用GPU**: 运行时 → 更改运行时类型 → T4 GPU
2. **高RAM（推荐）**: 运行时 → 更改运行时类型 → 高RAM
3. **按顺序执行**所有单元格
4. **配置API**: 在步骤8中填入你的API密钥

⏱️ **预计时间**: 首次运行约15-20分钟（后续约5-10分钟）

---

## 步骤 1️⃣: 检查GPU环境

In [ ]:
# 检查GPU
print('='*60)
print('🔍 检查GPU环境')
print('='*60)

!nvidia-smi

print('\n' + '='*60)
print('✅ GPU检查完成')
print('='*60)
print('\n⚠️  如果看不到GPU信息，请检查运行时设置')

## 步骤 2️⃣: 挂载Google Drive

💾 用于永久保存数据集和结果

In [ ]:
from google.colab import drive
import os

# 挂载Drive
drive.mount('/content/drive')

# 创建项目目录结构
PROJECT_DIR = '/content/drive/MyDrive/LayoutVLM_Project'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/results', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/datasets', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/blender_scripts', exist_ok=True)

print('\n' + '='*60)
print('✅ Google Drive已挂载')
print('='*60)
print(f'📁 项目目录: {PROJECT_DIR}')
print(f'📊 结果目录: {PROJECT_DIR}/results')
print(f'💾 数据集目录: {PROJECT_DIR}/datasets')
print('='*60)

## 步骤 3️⃣: 安装Blender 4.2.1

🔧 使用官方二进制包，支持完整的Python API

In [ ]:
import os

print('='*60)
print('🔧 安装Blender 4.2.1')
print('='*60)

# 1. 安装系统依赖
print('\n📦 步骤1/4: 安装系统依赖...')
!apt-get update -y -qq > /dev/null 2>&1
!apt-get install -y -qq xvfb libegl1-mesa libxrandr2 libxinerama1 libxxf86vm1 libxi6 wget > /dev/null 2>&1
print('   ✅ 系统依赖安装完成')

# 2. 下载Blender
print('\n📥 步骤2/4: 下载Blender 4.2.1...')
BLENDER_URL = "https://download.blender.org/release/Blender4.2/blender-4.2.1-linux-x64.tar.xz"
BLENDER_FILE = "blender-4.2.1-linux-x64.tar.xz"

if not os.path.exists(f'/content/{BLENDER_FILE}'):
    !wget -q --show-progress {BLENDER_URL}
    print('   ✅ 下载完成')
else:
    print('   ✅ Blender压缩包已存在')

# 3. 解压Blender
print('\n📂 步骤3/4: 解压Blender...')
if not os.path.exists('/content/blender-4.2.1-linux-x64'):
    !tar -xJf {BLENDER_FILE}
    print('   ✅ 解压完成')
else:
    print('   ✅ Blender已解压')

# 4. 设置环境变量
BLENDER = "/content/blender-4.2.1-linux-x64/blender"
os.environ['BLENDER_PATH'] = BLENDER

# 5. 验证安装
print('\n🔍 步骤4/4: 验证Blender安装...')
!$BLENDER --version

print('\n' + '='*60)
print('✅ Blender 4.2.1 安装完成')
print(f'📍 位置: {BLENDER}')
print('='*60)

## 步骤 4️⃣: 创建GPU配置脚本

⚡ 启用GPU加速渲染

In [ ]:
# 创建GPU配置脚本（基于你的set_cycles_gpu.py）
gpu_script = '''import bpy

# 配置Cycles渲染引擎使用GPU
prefs = bpy.context.preferences.addons["cycles"].preferences

# 尝试OPTIX，如果不支持则使用CUDA
try:
    prefs.compute_device_type = "OPTIX"
    print("✅ 使用OPTIX")
except Exception:
    prefs.compute_device_type = "CUDA"
    print("✅ 使用CUDA")

# 获取所有可用设备
prefs.get_devices()

# 启用所有GPU
gpu_count = 0
for dev in prefs.devices:
    try:
        dev.use = True
        if dev.type in ["CUDA", "OPTIX"]:
            gpu_count += 1
            print(f"   GPU {gpu_count}: {dev.name}")
    except:
        pass

# 设置场景使用GPU
bpy.context.scene.cycles.device = "GPU"

print(f"\\n[Cycles] 计算设备类型: {prefs.compute_device_type}")
print(f"[Cycles] 启用GPU数量: {gpu_count}")
print("✅ GPU渲染配置完成")
'''

# 保存到本地和Drive
with open('/content/set_cycles_gpu.py', 'w') as f:
    f.write(gpu_script)

import shutil
shutil.copy('/content/set_cycles_gpu.py', f'{PROJECT_DIR}/blender_scripts/set_cycles_gpu.py')

print('='*60)
print('✅ GPU配置脚本已创建')
print('='*60)
print('📄 本地: /content/set_cycles_gpu.py')
print(f'💾 备份: {PROJECT_DIR}/blender_scripts/set_cycles_gpu.py')
print('='*60)

## 步骤 5️⃣: 克隆LayoutVLM

📥 从GitHub获取最新代码

In [ ]:
import os

os.chdir('/content')

print('='*60)
print('📥 克隆LayoutVLM仓库')
print('='*60)

# 清理旧版本
if os.path.exists('LayoutVLM'):
    print('\n🗑️  清理旧版本...')
    !rm -rf LayoutVLM

# 克隆仓库
print('\n📦 正在克隆...')
!git clone -q https://github.com/sunfanyunn/LayoutVLM.git

os.chdir('/content/LayoutVLM')

print('\n' + '='*60)
print('✅ LayoutVLM已克隆')
print('='*60)
print(f'📍 位置: {os.getcwd()}')
print('\n📂 项目结构:')
!ls -1

## 步骤 6️⃣: 安装Python依赖到Blender

📦 **关键步骤**: 将依赖包安装到Blender的Python环境

In [ ]:
import os

BLENDER = os.environ['BLENDER_PATH']

print('='*60)
print('📦 安装Python依赖到Blender')
print('='*60)
print('\n⏱️  这可能需要3-5分钟，请耐心等待...\n')

# 创建依赖安装脚本
install_script = '''import sys
import subprocess

# 需要安装的包
packages = [
    "openai",
    "langchain",
    "langchain-openai",
    "langchain-core",
    "langchain-community",
    "numpy",
    "torch",
    "torchvision",
    "trimesh",
    "Pillow",
    "tiktoken",
    "pyyaml",
    "tqdm"
]

print(f"Python路径: {sys.executable}")
print(f"Python版本: {sys.version.split()[0]}")
print(f"\\n开始安装 {len(packages)} 个依赖包...\\n")

success = 0
failed = []

for i, package in enumerate(packages, 1):
    print(f"[{i}/{len(packages)}] 安装 {package}...", end=" ")
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "--no-warn-script-location", package],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        print("✅")
        success += 1
    except Exception as e:
        print(f"❌ ({str(e)[:30]}...)")
        failed.append(package)

print("\\n" + "="*60)
print(f"✅ 成功: {success}/{len(packages)}")
if failed:
    print(f"❌ 失败: {len(failed)} - {', '.join(failed)}")
print("="*60)
'''

# 保存安装脚本
with open('/tmp/install_deps_blender.py', 'w') as f:
    f.write(install_script)

# 使用Blender的Python执行安装
!$BLENDER --background --python /tmp/install_deps_blender.py

print('\n' + '='*60)
print('✅ 依赖安装完成')
print('='*60)

## 步骤 7️⃣: 编译 CUDA 扩展（核心步骤）

⚙️ **强制启用 CUDA 加速** - 完整诊断和编译流程

💡 本步骤会：
- 修复NumPy兼容性问题
- 安装CUDA编译工具（nvcc）
- 安装Python开发包（Python.h）
- **在Blender Python中编译CUDA扩展** ⭐

⏱️ 预计时间: 5-8分钟

📌 **注意**: 完成后请继续运行**步骤7.5**验证是否成功！

In [ ]:
import os
import sys

print('='*60)
print('⚙️  强制编译 CUDA 扩展 - 完整诊断')
print('='*60)
print('   🎯 目标: 确保 CUDA 扩展成功编译')
print('   ⏱️  预计时间: 5-8分钟\n')

BLENDER = os.environ.get('BLENDER_PATH', '/content/blender-4.2.1-linux-x64/blender')

# ========== 第1步: 修复 NumPy 兼容性 ==========
print('🔧 第1步: 修复 NumPy 兼容性')
min_box_file = '/content/LayoutVLM/third_party/Rotated_IoU/min_enclosing_box.py'
with open(min_box_file, 'r') as f:
    content = f.read()

if 'np.int)' in content or 'np.int,' in content:
    content = content.replace('.astype(np.int)', '.astype(int)')
    content = content.replace('dtype=np.int', 'dtype=int')
    with open(min_box_file, 'w') as f:
        f.write(content)
    print('   ✅ 已修复 min_enclosing_box.py')
else:
    print('   ✅ 无需修复（已是兼容版本）')

# ========== 第2步: 检查并安装 CUDA Toolkit ==========
print('\n🔧 第2步: 安装编译所需的系统工具\n')

# 检查 NVCC 是否存在
import subprocess
nvcc_found = False
try:
    result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
    if result.returncode == 0:
        nvcc_version = result.stdout.split('release')[-1].split(',')[0].strip()
        print(f'✅ NVCC 已安装: 版本 {nvcc_version}')
        nvcc_found = True
except:
    print('⚠️  NVCC 未找到')

# 如果未找到，安装 CUDA Toolkit
if not nvcc_found:
    print('\n📦 正在安装 CUDA Toolkit...')
    print('   这可能需要1-2分钟，请稍候...\n')
    !apt-get update -qq
    !apt-get install -y -qq nvidia-cuda-toolkit
    
    # 刷新环境变量和PATH
    print('\n🔄 刷新系统PATH...')
    import os
    # 添加常见的CUDA路径到PATH
    cuda_paths = [
        '/usr/local/cuda/bin',
        '/usr/local/cuda-12/bin',
        '/usr/local/cuda-11/bin',
        '/usr/bin'
    ]
    current_path = os.environ.get('PATH', '')
    for cuda_path in cuda_paths:
        if cuda_path not in current_path:
            os.environ['PATH'] = f"{cuda_path}:{current_path}"
            current_path = os.environ['PATH']
    
    print(f'   PATH 已更新')
    
    # 验证安装
    print('\n✅ 验证安装...')
    try:
        result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
        if result.returncode == 0:
            nvcc_version = result.stdout.split('release')[-1].split(',')[0].strip()
            print(f'✅ CUDA Toolkit 安装成功: 版本 {nvcc_version}')
            nvcc_found = True
        else:
            print('⚠️  安装后仍无法运行 nvcc')
            print(f'   错误: {result.stderr}')
    except Exception as e:
        print(f'❌ 验证失败: {e}')
    
    # 如果还是找不到，尝试手动查找
    if not nvcc_found:
        print('\n🔍 尝试手动查找 nvcc...')
        !which nvcc || find /usr -name nvcc -type f 2>/dev/null | head -5
else:
    print('   无需安装，继续下一步...')

# 安装 Python 开发头文件（编译 C++ 扩展必需）
print('\n📦 安装 Python 开发包（libpython3.11-dev）...')
print('   这是编译 C++ 扩展的必需组件...\n')
!apt-get install -y -qq libpython3.11-dev python3.11-dev

# 创建符号链接，让Blender Python能找到系统的Python.h
print('\n🔗 配置Python头文件路径...')
import os
blender_python_include = '/content/blender-4.2.1-linux-x64/4.2/python/include/python3.11'
system_python_include = '/usr/include/python3.11'

# 检查系统头文件是否存在
if os.path.exists(f'{system_python_include}/Python.h'):
    print(f'   ✅ 系统Python.h: {system_python_include}/Python.h')
    
    # 如果Blender的include目录不存在，创建它
    if not os.path.exists(blender_python_include):
        !mkdir -p {blender_python_include}
    
    # 创建符号链接（如果不存在）
    if not os.path.exists(f'{blender_python_include}/Python.h'):
        print(f'   🔗 创建符号链接...')
        !ln -sf {system_python_include}/* {blender_python_include}/
        print(f'   ✅ 符号链接已创建')
    else:
        print(f'   ✅ Python.h 已可用')
else:
    print(f'   ⚠️  系统Python.h未找到，尝试直接复制...')
    !cp -r /usr/include/python3.11/* {blender_python_include}/ 2>/dev/null || echo "复制失败"

# 验证
if os.path.exists(f'{blender_python_include}/Python.h'):
    print(f'\n✅ Python.h 验证成功: {blender_python_include}/Python.h')
else:
    print(f'\n❌ Python.h 仍然缺失')

print('\n✅ 系统工具安装完成')

# ========== 第3步: 完整诊断和编译 ==========
print('\n🔨 第3步: 完整诊断 + CUDA 编译\n')

blender_install_script = '''import sys
import subprocess
import os

print("=" * 70)
print("CUDA 扩展编译 - 完整诊断流程")
print("=" * 70)
print()

# ====== 阶段1: 环境检查 ======
print("📊 阶段1: 检查编译环境")
print("-" * 70)
print(f"Blender Python: {sys.executable}")
print(f"Python 版本: {sys.version.split()[0]}")

# 检查 CUDA
nvcc_found = False
try:
    result = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
    if result.returncode == 0:
        nvcc_version = result.stdout.split("release")[-1].split(",")[0].strip()
        print(f"✅ NVCC 版本: {nvcc_version}")
        nvcc_found = True
    else:
        print("⚠️  NVCC 不可用")
except:
    print("❌ NVCC 未找到")

if not nvcc_found:
    print("\\n⚠️  CUDA Toolkit 未安装！")
    print("   → 编译将会失败")
    print("   → 请确保在步骤7的第2步中成功安装了 CUDA Toolkit")

print()

# ====== 阶段2: 安装完整依赖 ======
print("📦 阶段2: 安装完整编译依赖")
print("-" * 70)

packages = [
    ("setuptools", "构建工具"),
    ("wheel", "打包工具"),
    ("ninja", "并行编译"),
    ("torch", "PyTorch 核心（含 CUDA 支持）"),
]

failed_packages = []
for pkg, desc in packages:
    print(f"   正在安装 {pkg} ({desc})...", end=" ", flush=True)
    try:
        # 安装 PyTorch 时指定 CUDA 版本
        if pkg == "torch":
            result = subprocess.run(
                [sys.executable, "-m", "pip", "install", "torch", "--index-url", 
                 "https://download.pytorch.org/whl/cu121"],  # CUDA 12.1
                capture_output=True,
                timeout=300
            )
        else:
            result = subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", pkg],
                capture_output=True,
                timeout=120
            )
        
        if result.returncode == 0:
            print("✅")
        else:
            print("❌")
            failed_packages.append(pkg)
            print(f"      错误: {result.stderr.decode()[:100]}")
    except subprocess.TimeoutExpired:
        print("⏱️ 超时")
        failed_packages.append(pkg)
    except Exception as e:
        print(f"❌ {str(e)[:50]}")
        failed_packages.append(pkg)

if failed_packages:
    print(f"\\n⚠️  依赖安装部分失败: {', '.join(failed_packages)}")
    print("   尝试继续编译...")
else:
    print("\\n✅ 所有依赖安装成功")

print()

# ====== 阶段3: 验证 PyTorch CUDA ======
print("🔍 阶段3: 验证 PyTorch CUDA 支持")
print("-" * 70)
try:
    import torch
    print(f"   PyTorch 版本: {torch.__version__}")
    print(f"   CUDA 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   CUDA 版本: {torch.version.cuda}")
        print(f"   GPU 设备: {torch.cuda.get_device_name(0)}")
    
    # 测试 cpp_extension
    try:
        from torch.utils.cpp_extension import BuildExtension, CUDAExtension
        print("   ✅ torch.utils.cpp_extension 可用")
    except ImportError as e:
        print(f"   ❌ cpp_extension 导入失败: {e}")
        print("   → 这是编译失败的根本原因！")
except Exception as e:
    print(f"   ❌ PyTorch 检查失败: {e}")

print()

# ====== 阶段4: 编译 CUDA 扩展 ======
print("🔨 阶段4: 编译 CUDA 扩展")
print("-" * 70)

os.chdir('/content/LayoutVLM/third_party/Rotated_IoU/cuda_op')

print("   编译命令: pip install .")
print("   工作目录: " + os.getcwd())

# 再次验证 nvcc（在编译前）
print("\\n   🔍 编译前最后检查 nvcc...")
nvcc_check = subprocess.run(["which", "nvcc"], capture_output=True, text=True)
if nvcc_check.returncode == 0:
    print(f"   ✅ nvcc 路径: {nvcc_check.stdout.strip()}")
    # 测试 nvcc 是否真的能运行
    nvcc_test = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
    if nvcc_test.returncode == 0:
        print(f"   ✅ nvcc 可执行")
    else:
        print(f"   ❌ nvcc 不可执行: {nvcc_test.stderr}")
else:
    print("   ❌ nvcc 不在 PATH 中")
    print("   → 尝试查找 nvcc...")
    find_nvcc = subprocess.run(["find", "/usr", "-name", "nvcc", "-type", "f"], 
                               capture_output=True, text=True, timeout=10)
    if find_nvcc.stdout:
        print(f"   找到的 nvcc 位置:\\n{find_nvcc.stdout}")

print("\\n   开始编译（详细模式）...\\n")

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-vv', '.'],
    capture_output=True,
    text=True,
    timeout=300
)

if result.returncode == 0:
    print("\\n" + "=" * 70)
    print("✅ 编译成功！")
    print("=" * 70)
else:
    print("\\n" + "=" * 70)
    print("❌ 编译失败")
    print("=" * 70)
    
    # 输出完整的错误信息
    full_output = result.stdout + "\\n" + result.stderr
    
    print("\\n📋 完整编译输出 (STDOUT):")
    print("-" * 70)
    print(result.stdout if result.stdout else "(无标准输出)")
    
    print("\\n❌ 完整错误输出 (STDERR):")
    print("-" * 70)
    print(result.stderr if result.stderr else "(无错误输出)")
    
    # 智能错误分析
    print("\\n🔍 智能错误分析:")
    print("-" * 70)
    if "No module named 'torch'" in full_output:
        print("   → PyTorch 未正确安装")
    if "nvcc" in full_output.lower() and ("not found" in full_output.lower() or "no such file" in full_output.lower()):
        print("   → NVCC (CUDA 编译器) 未找到或不在 PATH 中")
    if "cpp_extension" in full_output:
        print("   → torch.utils.cpp_extension 不可用")
    if "cuda" in full_output.lower() and "error" in full_output.lower():
        print("   → CUDA 相关错误（检查驱动和版本兼容性）")
    if "permission denied" in full_output.lower():
        print("   → 权限问题")
    if len(full_output.strip()) == 0:
        print("   → 无输出（可能超时或进程异常退出）")

print()

# ====== 阶段5: 验证编译结果 ======
print("✅ 阶段5: 验证编译结果")
print("-" * 70)

try:
    import sort_vertices
    print(f"   ✅ sort_vertices 导入成功")
    print(f"   📍 位置: {sort_vertices.__file__}")
    
    # 完整验证
    sys.path.insert(0, '/content/LayoutVLM')
    sys.path.insert(0, '/content/LayoutVLM/third_party/Rotated_IoU')
    
    from third_party.Rotated_IoU.oriented_iou_loss import cal_iou
    print(f"   ✅ oriented_iou_loss 可用")
    
    # 修复 matplotlib backend 问题（Blender环境需要非交互式backend）
    import os
    os.environ['MPLBACKEND'] = 'Agg'  # 使用非交互式backend
    
    from src.layoutvlm import constraints
    print(f"   ✅ ORIENTED_IOU_AVAILABLE = {constraints.ORIENTED_IOU_AVAILABLE}")
    
    if constraints.ORIENTED_IOU_AVAILABLE:
        print("\\n" + "=" * 70)
        print("🎉🎉🎉 CUDA 扩展完全成功！🎉🎉🎉")
        print("=" * 70)
        print("💡 性能: CUDA 比 CPU 快 5-10 倍")
    else:
        print("\\n⚠️  ORIENTED_IOU_AVAILABLE = False")
        print("   虽然模块导入成功，但标志未设置")
        
except ImportError as e:
    print(f"   ❌ 导入失败: {e}")
    print("\\n" + "=" * 70)
    print("💥 CUDA 扩展编译/导入失败")
    print("=" * 70)
    print("\\n🔧 下一步排查建议:")
    print("   1. 检查阶段3的 PyTorch CUDA 支持")
    print("   2. 查看阶段4的详细错误输出")
    print("   3. 确认 CUDA Toolkit 是否正确安装")
except Exception as e:
    print(f"   ❌ 验证异常: {e}")
    import traceback
    traceback.print_exc()
'''

with open('/tmp/install_cuda_full_diagnostic.py', 'w') as f:
    f.write(blender_install_script)

print('   开始完整诊断流程...\n')
print('='*70)
!$BLENDER --background --python /tmp/install_cuda_full_diagnostic.py

print('\n' + '='*60)
print('✅ 步骤7完成')
print('='*60)

## 步骤 7.8️⃣: 修复 CUDA 扩展导入问题（如果步骤11报错）

🔧 **修复 "No module named 'box_intersection_2d'" 错误**

💡 问题原因：CUDA扩展编译成功，但Python模块路径未正确设置

⏱️ 预计时间: 1分钟

In [ ]:
import os

BLENDER = os.environ.get('BLENDER_PATH', '/content/blender-4.2.1-linux-x64/blender')

print('='*60)
print('🔧 修复 CUDA 扩展导入问题')
print('='*60)
print()

# 创建修复脚本
fix_import_script = '''import sys
import os

print("=" * 70)
print("🔧 修复 box_intersection_2d 导入问题")
print("=" * 70)
print()

# 添加 Rotated_IoU 目录到 Python 路径
rotated_iou_path = '/content/LayoutVLM/third_party/Rotated_IoU'
if rotated_iou_path not in sys.path:
    sys.path.insert(0, rotated_iou_path)
    print(f"✅ 已添加路径: {rotated_iou_path}")

# 验证导入
print()
print("🔍 验证导入:")
print("-" * 70)

try:
    import box_intersection_2d
    print("✅ box_intersection_2d 导入成功")
    print(f"   位置: {box_intersection_2d.__file__}")
except ImportError as e:
    print(f"❌ box_intersection_2d 导入失败: {e}")
    print()
    print("🔍 查找 box_intersection_2d.py:")
    import subprocess
    result = subprocess.run(
        ['find', '/content/LayoutVLM/third_party/Rotated_IoU', '-name', 'box_intersection_2d.py'],
        capture_output=True, text=True
    )
    if result.stdout:
        print(f"   找到文件: {result.stdout.strip()}")
    else:
        print("   ❌ 文件不存在")

print()

try:
    from third_party.Rotated_IoU import oriented_iou_loss
    print("✅ oriented_iou_loss 导入成功")
    
    # 测试函数是否可用
    if hasattr(oriented_iou_loss, 'cal_iou'):
        print("✅ cal_iou 函数可用")
    if hasattr(oriented_iou_loss, 'cal_giou'):
        print("✅ cal_giou 函数可用")
    if hasattr(oriented_iou_loss, 'cal_my_giou'):
        print("✅ cal_my_giou 函数可用")
        
except ImportError as e:
    print(f"❌ oriented_iou_loss 导入失败: {e}")

print()

# 最终验证 constraints.py
print("🎯 最终验证 constraints.py:")
print("-" * 70)

# 添加 LayoutVLM 路径
sys.path.insert(0, '/content/LayoutVLM')

# 设置 matplotlib backend
os.environ['MPLBACKEND'] = 'Agg'

try:
    from src.layoutvlm import constraints
    print(f"✅ constraints 模块导入成功")
    print(f"📊 ORIENTED_IOU_AVAILABLE = {constraints.ORIENTED_IOU_AVAILABLE}")
    
    if constraints.ORIENTED_IOU_AVAILABLE:
        print()
        print("=" * 70)
        print("🎉 CUDA 扩展完全可用！")
        print("=" * 70)
    else:
        print()
        print("=" * 70)
        print("⚠️  ORIENTED_IOU_AVAILABLE = False")
        print("=" * 70)
        print()
        print("💡 这意味着:")
        print("   • CUDA 扩展编译成功")
        print("   • 但导入时出现问题")
        print("   • 将使用 CPU 后备方案")
        
except Exception as e:
    print(f"❌ constraints 导入失败: {e}")
    import traceback
    traceback.print_exc()

print()
print("=" * 70)
'''

with open('/tmp/fix_cuda_import.py', 'w') as f:
    f.write(fix_import_script)

print('🚀 开始修复...\n')
print('='*70)
!$BLENDER --background --python /tmp/fix_cuda_import.py

print('\n' + '='*60)
print('✅ 导入问题诊断完成')
print('='*60)
print()
print('💡 如果 ORIENTED_IOU_AVAILABLE = True:')
print('   → 需要更新步骤11的运行脚本')
print()
print('💡 如果仍然是 False:')
print('   → 使用 CPU 后备方案（性能略低但可用）')
print('='*60)

## 步骤 8️⃣: 准备数据集

💾 下载Objaverse资产数据集（约2.4GB）

## 步骤 7.5️⃣: 验证CUDA扩展（在Blender运行时）

🔍 **关键诊断**: 验证步骤7的编译是否真正成功

💡 本步骤会在**实际运行环境**（Blender Python）中测试：
- ✅ PyTorch是否支持CUDA
- ✅ sort_vertices模块是否能导入
- ✅ ORIENTED_IOU_AVAILABLE标志是否为True

⚠️ **重要说明**: 
- ❌ **不要在Colab Notebook中直接 `import sort_vertices`** 
- ✅ **必须通过Blender Python测试**（本单元格会自动执行）
- 原因：`sort_vertices` 只安装在Blender Python环境中

📌 **验证标准**: 
- ✅ 如果全部通过 → 继续步骤8（跳过7.6和7.7）
- ⚠️ 如果有失败 → 运行步骤7.6修复

In [ ]:
import os

BLENDER = os.environ.get('BLENDER_PATH', '/content/blender-4.2.1-linux-x64/blender')

print('='*60)
print('🔍 CUDA扩展在Blender运行时验证')
print('='*60)
print('\n💡 这一步会在实际运行LayoutVLM的环境中测试CUDA扩展\n')

# 创建验证脚本
verify_script = '''import sys
import os

print("=" * 70)
print("🔬 Blender Python 环境中的 CUDA 扩展验证")
print("=" * 70)
print()

print("📍 Python 信息:")
print(f"   Python 可执行文件: {sys.executable}")
print(f"   Python 版本: {sys.version.split()[0]}")
print()

# 添加路径
sys.path.insert(0, '/content/LayoutVLM')
sys.path.insert(0, '/content/LayoutVLM/third_party/Rotated_IoU')

print("📦 测试 PyTorch CUDA 支持:")
try:
    import torch
    print(f"   ✅ PyTorch 版本: {torch.__version__}")
    print(f"   ✅ CUDA 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   ✅ CUDA 版本: {torch.version.cuda}")
        print(f"   ✅ GPU 设备: {torch.cuda.get_device_name(0)}")
    else:
        print("   ⚠️  PyTorch 无法访问 CUDA！")
        print("      这可能是因为:")
        print("      1. Blender Python 没有安装 CUDA 版本的 PyTorch")
        print("      2. 环境变量未正确设置")
except ImportError as e:
    print(f"   ❌ 无法导入 PyTorch: {e}")
    print("   → 需要在 Blender Python 中重新安装 PyTorch")
except Exception as e:
    print(f"   ❌ PyTorch 检查失败: {e}")

print()

print("🔨 测试 CUDA 扩展:")
try:
    import sort_vertices
    print(f"   ✅ sort_vertices 导入成功")
    print(f"   📍 位置: {sort_vertices.__file__}")
except ImportError as e:
    print(f"   ❌ sort_vertices 导入失败: {e}")
    print("   → CUDA 扩展可能未正确编译或安装")
except Exception as e:
    print(f"   ❌ 导入异常: {e}")

print()

print("🎯 测试 constraints 模块:")
try:
    # 设置 matplotlib backend（避免 Blender 环境问题）
    os.environ['MPLBACKEND'] = 'Agg'
    
    from src.layoutvlm import constraints
    print(f"   ✅ constraints 模块导入成功")
    print(f"   📊 ORIENTED_IOU_AVAILABLE = {constraints.ORIENTED_IOU_AVAILABLE}")
    
    if constraints.ORIENTED_IOU_AVAILABLE:
        print("\\n" + "=" * 70)
        print("🎉 CUDA 扩展在 Blender 运行时完全可用！")
        print("=" * 70)
        print("✅ LayoutVLM 将使用 GPU 加速的 IoU 计算")
        print("💡 性能提升: 5-10x (相比 CPU 版本)")
    else:
        print("\\n" + "=" * 70)
        print("⚠️  CUDA 扩展标志为 False")
        print("=" * 70)
        print("\\n🔍 可能的原因:")
        print("   1. PyTorch CUDA 不可用")
        print("   2. sort_vertices 模块未正确导入")
        print("   3. oriented_iou_loss 导入失败")
        print("\\n💡 诊断建议:")
        print("   • 检查上方的 PyTorch CUDA 状态")
        print("   • 查看 sort_vertices 导入结果")
        print("   • 可能需要在 Blender Python 中重新安装 PyTorch")
        
except ImportError as e:
    print(f"   ❌ constraints 导入失败: {e}")
    import traceback
    print("\\n📋 详细错误信息:")
    traceback.print_exc()
except Exception as e:
    print(f"   ❌ 验证异常: {e}")
    import traceback
    print("\\n📋 详细错误信息:")
    traceback.print_exc()

print()
print("=" * 70)
print("✅ 验证完成")
print("=" * 70)
'''

with open('/tmp/verify_cuda_runtime.py', 'w') as f:
    f.write(verify_script)

print('🚀 开始验证...\n')
print('='*70)
!$BLENDER --background --python /tmp/verify_cuda_runtime.py

print('\n' + '='*60)
print('📊 验证结果总结')
print('='*60)
print('\n💡 如果看到 "CUDA 扩展在 Blender 运行时完全可用"：')
print('   ✅ 一切正常，可以继续下一步')
print('\n💡 如果看到 "CUDA 扩展标志为 False"：')
print('   ⚠️  需要修复 PyTorch 安装（见下方步骤 7.6）')
print('='*60)

## 步骤 7.5.1️⃣: 深度诊断CUDA问题（如果步骤11仍有警告）

🔬 **终极诊断**: 如果步骤11运行时仍然出现"No CUDA device available"警告，运行此单元格进行深度分析

💡 可能的原因：
1. **环境变量问题**: Blender运行时没有继承CUDA环境变量
2. **PyTorch重装问题**: 步骤7编译时重装了错误版本
3. **导入顺序问题**: constraints.py在某些情况下检测失败
4. **进程隔离**: xvfb-run可能隔离了GPU访问

⏱️ 预计时间: 2分钟

In [ ]:
import os

BLENDER = os.environ.get('BLENDER_PATH', '/content/blender-4.2.1-linux-x64/blender')

print('='*70)
print('🔬 深度诊断 - 找出CUDA失效的根本原因')
print('='*70)
print()

# 创建深度诊断脚本
deep_diagnostic = '''import sys
import os
import subprocess

print("=" * 80)
print("🔍 深度诊断 - CUDA 扩展失效原因分析")
print("=" * 80)
print()

# ====== 第1步: 环境信息 ======
print("📊 第1步: 完整环境信息")
print("-" * 80)
print(f"Python 可执行文件: {sys.executable}")
print(f"Python 版本: {sys.version}")
print(f"工作目录: {os.getcwd()}")
print()

print("环境变量:")
cuda_related_vars = ['CUDA_HOME', 'CUDA_PATH', 'CUDA_VISIBLE_DEVICES', 
                     'LD_LIBRARY_PATH', 'PATH']
for var in cuda_related_vars:
    value = os.environ.get(var, '<未设置>')
    if len(str(value)) > 100:
        value = str(value)[:100] + '...'
    print(f"  {var}: {value}")
print()

# ====== 第2步: PyTorch 详细检查 ======
print("🔍 第2步: PyTorch CUDA 详细状态")
print("-" * 80)

try:
    import torch
    print(f"✅ PyTorch 已安装")
    print(f"   版本: {torch.__version__}")
    print(f"   安装路径: {torch.__file__}")
    print()
    
    print("CUDA 状态:")
    print(f"   torch.cuda.is_available(): {torch.cuda.is_available()}")
    print(f"   torch.cuda.is_built(): {hasattr(torch.cuda, 'is_built') and torch.cuda.is_built()}")
    
    if hasattr(torch.version, 'cuda') and torch.version.cuda:
        print(f"   torch.version.cuda: {torch.version.cuda}")
    else:
        print(f"   torch.version.cuda: None (⚠️ 这是CPU版本!)")
    
    if torch.cuda.is_available():
        print(f"   GPU 数量: {torch.cuda.device_count()}")
        print(f"   当前设备: {torch.cuda.current_device()}")
        print(f"   设备名称: {torch.cuda.get_device_name(0)}")
    else:
        print("   ⚠️  CUDA 不可用")
        
        # 尝试诊断原因
        print()
        print("🔍 诊断 CUDA 不可用的原因:")
        
        # 检查是否是CPU版本
        if not hasattr(torch.version, 'cuda') or torch.version.cuda is None:
            print("   ❌ 检测到 CPU 版本的 PyTorch")
            print("   → 这是最常见的问题！")
            print("   → 解决方案: 运行步骤 7.6 重新安装 CUDA 版本")
        
        # 检查CUDA驱动
        try:
            result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
            if result.returncode == 0:
                print("   ✅ nvidia-smi 可执行（GPU驱动正常）")
            else:
                print("   ❌ nvidia-smi 失败")
                print(f"      错误: {result.stderr[:200]}")
        except Exception as e:
            print(f"   ❌ nvidia-smi 不可用: {e}")
        
        # 检查CUDA Toolkit
        try:
            result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
            if result.returncode == 0:
                print("   ✅ nvcc 可执行（CUDA Toolkit 已安装）")
            else:
                print("   ❌ nvcc 失败")
        except Exception as e:
            print(f"   ❌ nvcc 不可用: {e}")
            
except ImportError as e:
    print(f"❌ 无法导入 PyTorch: {e}")
    print("   → 需要重新安装 PyTorch")

print()

# ====== 第3步: sort_vertices 检查 ======
print("🔨 第3步: sort_vertices 扩展状态")
print("-" * 80)

try:
    import sort_vertices
    print(f"✅ sort_vertices 已导入")
    print(f"   位置: {sort_vertices.__file__}")
    
    # 尝试调用函数（如果有的话）
    if hasattr(sort_vertices, '__version__'):
        print(f"   版本: {sort_vertices.__version__}")
    
    print()
    
except ImportError as e:
    print(f"❌ sort_vertices 导入失败: {e}")
    print("   → CUDA 扩展未正确编译或安装")
    print()

# ====== 第4步: constraints.py 检查 ======
print("🎯 第4步: constraints.py 中的 CUDA 检测逻辑")
print("-" * 80)

# 添加路径
sys.path.insert(0, '/content/LayoutVLM')
sys.path.insert(0, '/content/LayoutVLM/third_party/Rotated_IoU')

# 设置 matplotlib backend
os.environ['MPLBACKEND'] = 'Agg'

try:
    # 先测试单独的 oriented_iou_loss
    print("测试 oriented_iou_loss 导入:")
    try:
        from third_party.Rotated_IoU.oriented_iou_loss import cal_iou
        print("   ✅ oriented_iou_loss.cal_iou 可导入")
        
        # 测试是否依赖CUDA
        import torch
        if not torch.cuda.is_available():
            print("   ⚠️  但 torch.cuda.is_available() = False")
            print("   → oriented_iou_loss 可能在运行时失败")
        
    except ImportError as e:
        print(f"   ❌ oriented_iou_loss 导入失败: {e}")
        print("   → 这会导致 ORIENTED_IOU_AVAILABLE = False")
    except Exception as e:
        print(f"   ❌ oriented_iou_loss 异常: {e}")
    
    print()
    
    # 导入 constraints 并检查标志
    print("测试 constraints 模块:")
    from src.layoutvlm import constraints
    
    print(f"   ✅ constraints 模块已导入")
    print(f"   📊 ORIENTED_IOU_AVAILABLE = {constraints.ORIENTED_IOU_AVAILABLE}")
    print()
    
    if not constraints.ORIENTED_IOU_AVAILABLE:
        print("❌ ORIENTED_IOU_AVAILABLE = False")
        print()
        print("🔍 可能的原因:")
        print("   1. PyTorch 是 CPU 版本（最可能）")
        print("   2. sort_vertices 编译失败")
        print("   3. oriented_iou_loss 导入时发生异常")
        print("   4. constraints.py 中的 try-except 捕获了错误")
        print()
        print("💡 解决方案:")
        print("   → 运行步骤 7.6 重新安装 PyTorch CUDA 版本")
        print("   → 确保使用命令: pip install torch==2.5.1 --index-url ...")
    else:
        print("✅ ORIENTED_IOU_AVAILABLE = True")
        print("   → CUDA 扩展应该可以正常工作")
        print()
        print("⚠️  如果步骤11仍有警告，可能是:")
        print("   1. 运行时环境与验证环境不同")
        print("   2. xvfb-run 导致的环境隔离")
        print("   3. 多进程导致的环境变量丢失")
    
except ImportError as e:
    print(f"❌ constraints 导入失败: {e}")
    import traceback
    print()
    print("详细错误:")
    traceback.print_exc()
except Exception as e:
    print(f"❌ 验证异常: {e}")
    import traceback
    print()
    print("详细错误:")
    traceback.print_exc()

print()

# ====== 第5步: 读取 constraints.py 源码 ======
print("📄 第5步: 分析 constraints.py 源码")
print("-" * 80)

constraints_file = '/content/LayoutVLM/src/layoutvlm/constraints.py'
if os.path.exists(constraints_file):
    print(f"读取文件: {constraints_file}")
    print()
    
    with open(constraints_file, 'r') as f:
        content = f.read()
    
    # 查找 ORIENTED_IOU_AVAILABLE 的定义
    lines = content.split('\\n')
    found_flag = False
    for i, line in enumerate(lines):
        if 'ORIENTED_IOU_AVAILABLE' in line or 'oriented_iou_loss' in line.lower():
            start = max(0, i - 3)
            end = min(len(lines), i + 4)
            if not found_flag:
                print("ORIENTED_IOU_AVAILABLE 标志定义:")
                print("-" * 80)
                found_flag = True
            for j in range(start, end):
                marker = '>>> ' if j == i else '    '
                print(f"{marker}{j+1:4d}: {lines[j]}")
            print()
    
    if not found_flag:
        print("⚠️  未找到 ORIENTED_IOU_AVAILABLE 定义")
        print("   文件可能已被修改")
else:
    print(f"❌ constraints.py 不存在: {constraints_file}")

print()

# ====== 总结 ======
print("=" * 80)
print("📋 诊断总结")
print("=" * 80)
print()

# 读取当前状态
try:
    import torch
    cuda_available = torch.cuda.is_available()
    torch_version = torch.__version__
    has_cuda_version = hasattr(torch.version, 'cuda') and torch.version.cuda is not None
    
    from src.layoutvlm import constraints
    oriented_iou_flag = constraints.ORIENTED_IOU_AVAILABLE
    
    print("当前状态:")
    print(f"  • PyTorch 版本: {torch_version}")
    print(f"  • PyTorch CUDA 版本: {torch.version.cuda if has_cuda_version else 'None (CPU版本)'}")
    print(f"  • torch.cuda.is_available(): {cuda_available}")
    print(f"  • ORIENTED_IOU_AVAILABLE: {oriented_iou_flag}")
    print()
    
    if not cuda_available:
        print("🔴 主要问题: PyTorch 不支持 CUDA")
        print()
        print("根本原因:")
        if not has_cuda_version:
            print("  ❌ 安装了 CPU 版本的 PyTorch")
            print()
            print("✅ 解决方案:")
            print("  1. 运行步骤 7.6 修复 PyTorch")
            print("  2. 或在 Blender Python 中手动运行:")
            print()
            print("     pip uninstall -y torch torchvision")
            print("     pip install torch==2.5.1 torchvision==0.20.1 \\\\")
            print("       --index-url https://download.pytorch.org/whl/cu121")
        else:
            print("  ⚠️  虽然是 CUDA 版本，但 is_available() = False")
            print()
            print("可能原因:")
            print("  • GPU 驱动版本不兼容")
            print("  • CUDA 版本不匹配")
            print("  • 环境变量未设置")
    elif not oriented_iou_flag:
        print("🟡 警告: PyTorch CUDA 可用，但 ORIENTED_IOU_AVAILABLE = False")
        print()
        print("可能原因:")
        print("  • oriented_iou_loss 导入失败")
        print("  • sort_vertices 编译问题")
        print()
        print("✅ 解决方案:")
        print("  1. 检查上方第3步的 sort_vertices 状态")
        print("  2. 重新运行步骤 7（重新编译扩展）")
    else:
        print("🟢 状态正常: CUDA 完全可用")
        print()
        print("如果步骤11仍有警告:")
        print("  • 可能是运行时环境问题")
        print("  • 尝试不使用 xvfb-run 直接运行")
        print("  • 检查是否有多进程环境变量丢失")
        
except Exception as e:
    print(f"❌ 无法生成总结: {e}")

print()
print("=" * 80)
'''

with open('/tmp/deep_diagnostic_cuda.py', 'w') as f:
    f.write(deep_diagnostic)

print('🚀 开始深度诊断...\n')
print('='*70)
!$BLENDER --background --python /tmp/deep_diagnostic_cuda.py

print('\n' + '='*70)
print('✅ 深度诊断完成')
print('='*70)
print()
print('💡 请仔细查看上方输出，特别是:')
print('   1. "PyTorch CUDA 详细状态" - 检查是否是CPU版本')
print('   2. "constraints.py 中的 CUDA 检测逻辑" - 查看失败原因')
print('   3. "诊断总结" - 获取具体的解决方案')
print('='*70)

## 步骤 7.6️⃣: 修复PyTorch CUDA（如果需要）

⚠️ **仅在步骤7.5显示CUDA不可用时运行此步骤**

💡 此步骤会确保Blender Python环境中的PyTorch支持CUDA

In [ ]:
import os

BLENDER = os.environ.get('BLENDER_PATH', '/content/blender-4.2.1-linux-x64/blender')

print('='*60)
print('🔧 修复 Blender Python 中的 PyTorch CUDA')
print('='*60)
print('\n⏱️  预计时间: 2-3分钟\n')

# 创建修复脚本
fix_script = '''import sys
import subprocess

print("=" * 70)
print("🔨 重新安装 PyTorch (CUDA 版本)")
print("=" * 70)
print()

print(f"📍 Python: {sys.executable}")
print()

# 卸载现有 PyTorch
print("🗑️  卸载现有 PyTorch...")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision"], 
               capture_output=True)
print("   ✅ 卸载完成")
print()

# 重新安装 CUDA 版本的 PyTorch
print("📦 安装 PyTorch 2.5.1+cu121...")
print("   这可能需要1-2分钟，请稍候...")
print()

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", 
     "torch==2.5.1", "torchvision==0.20.1",
     "--index-url", "https://download.pytorch.org/whl/cu121"],
    capture_output=True,
    text=True,
    timeout=300
)

if result.returncode == 0:
    print("✅ PyTorch 安装成功")
else:
    print("❌ 安装失败:")
    print(result.stderr[:500])
print()

# 验证安装
print("🔍 验证 PyTorch CUDA 支持:")
try:
    import torch
    print(f"   ✅ PyTorch 版本: {torch.__version__}")
    print(f"   ✅ CUDA 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   ✅ CUDA 版本: {torch.version.cuda}")
        print(f"   ✅ GPU 设备: {torch.cuda.get_device_name(0)}")
        print()
        print("=" * 70)
        print("🎉 PyTorch CUDA 修复成功！")
        print("=" * 70)
    else:
        print("   ⚠️  CUDA 仍然不可用")
        print()
        print("=" * 70)
        print("❌ 修复失败")
        print("=" * 70)
        print("\\n💡 可能需要:")
        print("   1. 检查 GPU 运行时是否启用")
        print("   2. 重启 Colab 运行时")
except Exception as e:
    print(f"   ❌ 验证失败: {e}")
'''

with open('/tmp/fix_pytorch_cuda.py', 'w') as f:
    f.write(fix_script)

print('🚀 开始修复...\n')
print('='*70)
!$BLENDER --background --python /tmp/fix_pytorch_cuda.py

print('\n' + '='*60)
print('✅ 修复步骤完成')
print('='*60)
print('\n💡 完成后，请重新运行步骤 7.5 验证')
print('='*60)

## 步骤 7.7️⃣: 修复API Prompt问题

🔧 **解决 "LLM returned natural language instead of code" 错误**

💡 这个问题通常由以下原因引起：
1. **API模型选择**: 某些转接API的GPT-4o可能不支持Vision
2. **Prompt不够明确**: LLM把任务理解为对话而非代码生成
3. **图片格式问题**: 图片编码或大小超出限制

**推荐解决方案**：

In [ ]:
print('='*60)
print('🔧 API Prompt 增强修复')
print('='*60)
print()

# 修改 short_prompt.py，增强指令明确性
prompt_fix = '''
import os

# 备份原始文件
prompt_file = '/content/LayoutVLM/prompts/layoutvlm/short_prompt.py'
backup_file = '/content/LayoutVLM/prompts/layoutvlm/short_prompt.py.backup'

if not os.path.exists(backup_file):
    import shutil
    shutil.copy(prompt_file, backup_file)
    print('✅ 已备份原始 prompt 文件')
else:
    print('✅ 备份文件已存在')

# 读取文件
with open(prompt_file, 'r') as f:
    content = f.read()

# 检查是否已经修改过
if '**CRITICAL INSTRUCTION**' in content:
    print('✅ Prompt 已经增强过，无需修改')
else:
    # 在 program_prompt 的开头添加强调指令
    enhanced_instruction = '''
**CRITICAL INSTRUCTION**: You are a coding agent that MUST respond ONLY with valid Python code.
DO NOT write any natural language explanations outside the code block.
DO NOT ask questions or request clarifications.
ALWAYS wrap your code in ```python ... ``` tags.
If you cannot see the images clearly, make your best effort to infer the layout from the text description.

'''
    
    # 查找 program_prompt = """ 的位置
    if 'program_prompt = """' in content:
        content = content.replace(
            'program_prompt = """',
            'program_prompt = """' + enhanced_instruction
        )
        
        # 写回文件
        with open(prompt_file, 'w') as f:
            f.write(content)
        
        print('✅ Prompt 增强成功！')
        print('\\n📋 添加的增强指令:')
        print(enhanced_instruction)
    else:
        print('⚠️  未找到 program_prompt 定义，可能文件结构已变化')

print()
print('=' * 60)
print('💡 修复说明')
print('=' * 60)
print('\\n✅ 已增强的功能:')
print('   1. 明确要求只输出代码（不输出解释）')
print('   2. 禁止询问问题或请求澄清')
print('   3. 强制使用代码块格式')
print('   4. 即使看不清图片也要尝试生成代码')
print()
print('💡 如果仍然失败，可能的原因:')
print('   1. API 不支持 Vision 功能')
print('   2. 图片太大（已经自动压缩到 1024x1024, 质量85）')
print('   3. 需要更换支持 Vision 的 API 提供商')
print()
print('🔧 高级修复选项:')
print('   • 降低图片质量（修改 layoutvlm.py 中的 quality 参数）')
print('   • 减小图片尺寸（修改 max_size 参数）')
print('   • 使用 no_image 模式（不使用图片，仅文本）')
print('=' * 60)
'''

# 执行修复
exec(prompt_fix)

print()
print('='*60)
print('✅ API Prompt 修复完成')
print('='*60)
print('\\n💡 下次运行步骤11时，将使用增强的 Prompt')
print('='*60)

---

## 🔧 故障排查总结

### ⚠️ 问题1: "LLM returned natural language instead of code"

**症状**: API返回对话式文本而非Python代码
```
"It looks like you've uploaded two additional images..."
"Could you clarify how you would like these images..."
```

**解决方案**:
1. ✅ **运行步骤 7.7** - 增强Prompt指令
2. 🔍 **检查API配置** - 确认使用支持Vision的模型（GPT-4o或GPT-4-vision）
3. 📸 **降低图片质量** - 如果API有size限制，修改`layoutvlm.py`第97行：
   ```python
   # 从 quality=85 改为 quality=70
   encoded_images = [self.encode_image(image_path, quality=70) for ...]
   ```
4. 🚨 **最后手段** - 使用no_image模式（性能会下降）：
   ```python
   # 在步骤11的运行脚本中修改
   mode="no_image"  # 不使用图片
   ```

**根本原因**:
- 某些转接API的GPT-4o可能不完全兼容Vision功能
- Prompt需要更强的约束来避免对话式回复
- 图片编码可能超出API token限制

---

### ⚠️ 问题2: "WARNING: No CUDA device available"

**症状**: 运行时显示多次CUDA警告，未使用GPU加速
```
WARNING: No CUDA device available, skipping important oriented IoU loss function.
```

**解决方案**:
1. ✅ **运行步骤 7.5** - 验证Blender运行时的CUDA状态
2. 🔧 **如果CUDA不可用** - 运行步骤 7.6 修复PyTorch
3. 🔍 **验证编译产物**:
   ```bash
   # 检查.so文件是否存在
   ls -lh /content/blender-4.2.1-linux-x64/4.2/python/lib/python3.11/site-packages/sort_vertices*.so
   ```
4. 🔄 **重启运行时** - 如果修复后仍失败，尝试重启Colab

**根本原因**:
- 步骤6安装依赖时未安装CUDA版本的PyTorch
- Blender Python环境与系统Python环境隔离
- 需要在Blender Python中单独安装PyTorch+CUDA

**性能影响**:
- ❌ 没有CUDA: 使用Shapely CPU后备，速度慢5-10倍
- ✅ 使用CUDA: GPU加速的Rotated IoU，约束优化快5-10倍

---

### 💡 调试技巧

**查看详细日志**:
```python
# 在步骤11的运行脚本中添加
import logging
logging.basicConfig(level=logging.DEBUG)
```

**测试单张图片**:
```python
# 减少重试次数，快速失败
MAX_ATTEMPTS = 1  # 默认是3
```

**使用benchmark样例**:
```python
# 复制已验证的场景配置
!cp benchmark_tasks/living_room/living_room_0.json scene_config.json
```

---

## 📋 步骤7系列总结

### ✅ 已完成的修复步骤

| 步骤 | 名称 | 作用 | 必需性 |
|------|------|------|--------|
| **步骤7** | 编译CUDA扩展 | 安装工具+编译.so文件 | ⭐ **必须运行** |
| **步骤7.5** | 验证CUDA扩展 | 检查编译是否成功 | 🔍 **强烈推荐** |
| **步骤7.6** | 修复PyTorch | 重装CUDA版PyTorch | ⚠️ **按需运行** |
| **步骤7.7** | 增强API Prompt | 避免自然语言回复 | 💡 **推荐运行** |

### 🎯 执行策略

**首次运行**:
```
步骤7 → 步骤7.5 → [如果失败]步骤7.6 → 步骤7.7 → 步骤8
```

**快速验证**（如果之前成功过）:
```
步骤7.5 → [如果通过]直接跳到步骤8
```

### 🔍 成功标准

**步骤7成功**:
- ✅ 编译输出: "✅ 编译成功！"
- ✅ 文件存在: `sort_vertices.cpython-311-x86_64-linux-gnu.so`

**步骤7.5成功**:
- ✅ PyTorch CUDA可用: `True`
- ✅ sort_vertices导入: 成功
- ✅ ORIENTED_IOU_AVAILABLE: `True`

**最终验证**（步骤11运行时）:
- ✅ 无"WARNING: No CUDA device available"警告
- ✅ 优化速度: ~1-2分钟（而非5-10分钟）

---

In [ ]:
import os

DATASET_PATH = f'{PROJECT_DIR}/datasets/dataset.zip'

print('='*60)
print('💾 准备数据集')
print('='*60)

# 检查Drive中是否已有数据集
if os.path.exists(DATASET_PATH):
    print('\n✅ 数据集已存在于Drive，直接复制...')
    !cp {DATASET_PATH} /content/LayoutVLM/dataset.zip
    print('   ✅ 复制完成')
else:
    print('\n📥 首次运行，正在下载数据集（约2.4GB）...')
    print('   这可能需要3-5分钟...\n')
    
    !pip install -q gdown
    !gdown 1WGbj8gWn-f-BRwqPKfoY06budBzgM0pu -O /content/LayoutVLM/dataset.zip
    
    print('\n💾 备份数据集到Drive（下次运行更快）...')
    !cp /content/LayoutVLM/dataset.zip {DATASET_PATH}
    print('   ✅ 备份完成')

# 解压数据集
print('\n📂 解压数据集...')
!mkdir -p /content/LayoutVLM/data
!unzip -q /content/LayoutVLM/dataset.zip -d /content/LayoutVLM/data/

# 检查解压结果
print('\n📊 数据集内容:')
!ls -lh /content/LayoutVLM/data/ | head -10

print('\n' + '='*60)
print('✅ 数据集准备完成')
print('='*60)

## 步骤 9️⃣: 配置转接API

🔑 **重要**: 填入你的API密钥和配置

In [ ]:
import os

print('='*60)
print('🔑 配置转接API')
print('='*60)

# ⚠️⚠️⚠️ 在这里填入你的配置 ⚠️⚠️⚠️
API_KEY = "sk-YOUR_API_KEY_HERE"  # 替换成你的API密钥！
BASE_URL = "https://chat.cloudapi.vip/v1/"
MODEL_NAME = "gpt-4o"  # 或其他支持vision的模型

# 设置环境变量
os.environ['OPENAI_API_KEY'] = API_KEY
os.environ['OPENAI_BASE_URL'] = BASE_URL

print('\n📋 API配置信息:')
print(f'   🔑 API Key: {API_KEY[:15]}...')
print(f'   🌐 Base URL: {BASE_URL}')
print(f'   🤖 Model: {MODEL_NAME}')

# 测试API连接
print('\n🧪 测试API连接...')
try:
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": "测试"}],
        max_tokens=5
    )
    print('   ✅ API连接成功')
    print(f'   📨 响应: {response.choices[0].message.content}')
except Exception as e:
    print(f'   ❌ API测试失败: {e}')
    print('   ⚠️  请检查API Key和Base URL是否正确')

print('\n' + '='*60)
print('✅ API配置完成')
print('='*60)

## 步骤 🔟: 创建场景配置

📝 定义要生成的3D场景

In [ ]:
import json
import os

os.chdir('/content/LayoutVLM')

print('='*60)
print('📝 创建场景配置')
print('='*60)

# 创建示例场景
scene_config = {
    "task_description": "Create a cozy living room with modern furniture",
    "layout_criteria": "Natural furniture arrangement with good flow and spacing. Sofa against the wall, coffee table in front of it.",
    "boundary": {
        "floor_vertices": [
            [0, 0, 0],
            [6, 0, 0],
            [6, 0, 5],
            [0, 0, 5]
        ],
        "wall_height": 3.0
    },
    "assets": {
        # 这里会由LayoutVLM根据数据集自动填充
        # 或者你可以从benchmark_tasks中复制示例
    }
}

# 保存配置
with open('scene_config.json', 'w') as f:
    json.dump(scene_config, f, indent=2)

print('\n✅ 场景配置已创建')
print('\n📄 配置内容:')
print(json.dumps(scene_config, indent=2))

print('\n' + '='*60)
print('💡 提示: 你可以查看 benchmark_tasks/ 目录')
print('   获取更多场景配置示例')
print('='*60)

# 显示可用的示例
print('\n📚 可用的benchmark示例:')
!find benchmark_tasks -name "*.json" | head -5

## 步骤 1️⃣1️⃣: 运行LayoutVLM

🚀 **核心步骤**: 使用Blender Python运行LayoutVLM生成布局

⏱️ 预计时间: 5-15分钟（取决于场景复杂度）

🔧 **已修复的问题**:
1. ✅ **CUDA隔离**: 不使用 `xvfb-run`，保留GPU访问
2. ✅ **matplotlib backend**: 清除Colab环境变量污染
3. ✅ **CUDA扩展导入**: 添加 `Rotated_IoU` 到Python路径

💡 如果看到 "No module named 'box_intersection_2d'" 警告：
   → 不影响运行，会自动使用CPU后备方案
   → 或运行步骤7.8诊断并修复

In [ ]:
import os

BLENDER = os.environ['BLENDER_PATH']
os.chdir('/content/LayoutVLM')

print('='*60)
print('🚀 运行LayoutVLM')
print('='*60)
print('\n⏱️  这可能需要5-15分钟，请耐心等待...')
print('💡 你可以在下方看到实时进度\n')

# 创建运行脚本
run_script = f'''import os
import sys

# ========== 修复 matplotlib backend 问题 ==========
# Colab 的 MPLBACKEND 环境变量与 Blender 不兼容
# 必须在导入任何模块前设置
if 'MPLBACKEND' in os.environ:
    del os.environ['MPLBACKEND']
os.environ['MPLBACKEND'] = 'Agg'  # 使用非交互式 backend

# 设置环境变量
os.environ["OPENAI_API_KEY"] = "{API_KEY}"
os.environ["OPENAI_BASE_URL"] = "{BASE_URL}"

# ========== 修复 CUDA 扩展导入路径 ==========
# 添加 Rotated_IoU 到 Python 路径，确保 box_intersection_2d 可导入
sys.path.insert(0, "/content/LayoutVLM/third_party/Rotated_IoU")
sys.path.insert(0, "/content/LayoutVLM")

# 设置命令行参数
sys.argv = [
    "main.py",
    "--scene_json_file", "scene_config.json",
    "--openai_api_key", "{API_KEY}",
    "--save_dir", "{PROJECT_DIR}/results"
]

print("="*60)
print("🎨 LayoutVLM 开始生成布局")
print("="*60)
print()

# 执行main.py
try:
    with open("/content/LayoutVLM/main.py", "r") as f:
        exec(f.read())
    print()
    print("="*60)
    print("✅ LayoutVLM执行完成")
    print("="*60)
except Exception as e:
    print()
    print("="*60)
    print(f"❌ 执行出错: {{e}}")
    print("="*60)
    import traceback
    traceback.print_exc()
'''

# 保存运行脚本
with open('/tmp/run_layoutvlm.py', 'w') as f:
    f.write(run_script)

print('='*60)
print('开始执行...')
print('='*60)
print()

# ========== 修复: 直接运行，不使用 xvfb-run ==========
# 原因: xvfb-run 可能导致 GPU 访问被阻断，CUDA 环境变量丢失
# 诊断结果: CUDA 在 Blender Python 中完全可用，但 xvfb-run 导致隔离

# 设置 DISPLAY 环境变量（避免 Blender 尝试打开窗口）
import os
os.environ['DISPLAY'] = ':99'

# 启动 Xvfb 作为后台服务（如果需要图形支持）
print('🖥️  启动虚拟显示服务...')
import subprocess
xvfb_process = subprocess.Popen(
    ['Xvfb', ':99', '-screen', '0', '1024x768x24'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
import time
time.sleep(2)  # 等待 Xvfb 启动
print('   ✅ 虚拟显示已启动\n')

# 直接运行 Blender（不通过 xvfb-run）
# 这样可以保留完整的 CUDA 环境
!$BLENDER --background --python /tmp/run_layoutvlm.py

# 清理 Xvfb 进程
xvfb_process.terminate()
xvfb_process.wait()

print()
print('='*60)
print('✅ 运行完成！')
print('='*60)
print(f'📁 结果保存在: {PROJECT_DIR}/results')
print()
print('💡 如果仍有 CUDA 警告，请检查上方日志')
print('='*60)

## 步骤 1️⃣2️⃣: 查看生成结果

📊 查看生成的布局数据和渲染图片

In [ ]:
import os
import json
from IPython.display import Image, display
import glob

result_dir = f'{PROJECT_DIR}/results'

print('='*60)
print('📊 查看生成结果')
print('='*60)

# 列出所有生成的文件
print('\n📁 生成的文件:')
!ls -lh {result_dir}

# 读取布局JSON
layout_file = f'{result_dir}/layout.json'

if os.path.exists(layout_file):
    print('\n' + '='*60)
    print('📋 布局数据 (layout.json):')
    print('='*60)
    
    with open(layout_file, 'r') as f:
        layout = json.load(f)
    
    # 显示简要信息
    print(f'\n✅ 成功生成 {len(layout)} 个物体的布局')
    
    # 显示前几个物体的信息
    print('\n前3个物体的信息:')
    for i, (obj_id, obj_info) in enumerate(list(layout.items())[:3], 1):
        print(f'\n{i}. {obj_id}:')
        print(f'   位置: {obj_info.get("position", "N/A")}')
        print(f'   旋转: {obj_info.get("rotation", "N/A")}')
        if len(layout) > 3 and i == 3:
            print(f'\n... 还有 {len(layout) - 3} 个物体')
    
    # 完整JSON预览
    print('\n完整JSON数据（前500字符）:')
    json_str = json.dumps(layout, indent=2)
    print(json_str[:500])
    if len(json_str) > 500:
        print(f'\n... (还有 {len(json_str) - 500} 字符)')
    
else:
    print('\n❌ 未找到 layout.json 文件')
    print('   请检查运行日志中的错误信息')

# 查找并显示渲染图片
print('\n' + '='*60)
print('🖼️  渲染图片:')
print('='*60)

image_files = glob.glob(f'{result_dir}/*.png') + glob.glob(f'{result_dir}/*.jpg')

if image_files:
    print(f'\n找到 {len(image_files)} 张图片:\n')
    for img_path in image_files[:5]:  # 最多显示5张
        print(f'📷 {os.path.basename(img_path)}')
        try:
            display(Image(filename=img_path, width=600))
            print()
        except:
            print(f'   ⚠️  无法显示图片')
    
    if len(image_files) > 5:
        print(f'\n... 还有 {len(image_files) - 5} 张图片')
else:
    print('\n⚠️  未找到渲染图片')
    print('   可能渲染功能未启用或发生错误')

print('\n' + '='*60)
print('✅ 结果查看完成')
print('='*60)
print(f'\n💾 所有结果已保存在: {result_dir}')
print('   你可以在Google Drive中查看完整结果')

---

## 🔧 故障排查

### 常见问题

#### 1. API连接失败
- 检查API Key是否正确
- 确认Base URL格式正确（包含 `/v1/`）
- 检查账户余额

#### 2. 内存不足
- 尝试使用高RAM运行时
- 减小场景复杂度

#### 3. GPU相关错误
- 确认已启用GPU运行时
- 检查CUDA版本兼容性

#### 4. 依赖安装失败
- 重新运行步骤6
- 检查网络连接

### 重新运行

如果需要重新运行：
1. **完全重启**: 运行时 → 重启运行时
2. **从头执行**: 按顺序重新运行所有单元格
3. **数据集会从Drive快速恢复**（无需重新下载）

---

## 📚 参考资源

- **论文**: [LayoutVLM: Differentiable Optimization of 3D Layout via Vision-Language Models](https://arxiv.org/abs/2412.02193)
- **GitHub**: https://github.com/sunfanyunn/LayoutVLM
- **项目主页**: https://ai.stanford.edu/~sunfanyun/layoutvlm/
- **CVPR 2025**: Proceedings (June 2025)

---

## 💡 下一步

1. **尝试不同场景**: 修改`scene_config.json`
2. **使用benchmark**: 复制`benchmark_tasks`中的示例
3. **调整参数**: 修改布局标准和边界
4. **自定义资产**: 添加自己的3D模型

---